# 06 - Rejection and calibration

**Goal:** calibrate confidence and reject uncertain or unsupported images using a combination of:
- temperature scaling on logits
- max-logit or margin thresholding
- embedding distance to class prototypes

The final app-side decision will be one of:
- `possible_match`
- `uncertain`
- `unsupported_or_unclear`


In [1]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import mobilenet_v3_small
import torch_directml

In [2]:


PROJECT_ROOT = Path.cwd().parent
ARTIFACTS = PROJECT_ROOT / "agrilens-training/artifacts"
ARTIFACTS.mkdir(exist_ok=True)

dml = torch_directml.device()
print("DirectML device:", torch_directml.device_name(0))


DirectML device: AMD Radeon RX 5700 


In [3]:
def seed_everything(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(42)


### Step 1 - load model + class map + validation/test splits

Inference uses batches of 8 and disables gradient tracking for logits and embeddings.
If DirectML runs out of memory, the current batch is retried on CPU and subsequent
batches stay on CPU. For a clean run after a GPU failure, restart the kernel and run
all cells. Set `inference_device = torch.device("cpu")` below to force CPU inference.


In [4]:
class_map_path = ARTIFACTS / "class_map.json"
manifest_path = ARTIFACTS / "manifest_v2.csv"
preprocess_path = ARTIFACTS / "preprocess_contract.json"
checkpoint_path = ARTIFACTS / "mobilenetv3_finetune_best.pt"

for p in [class_map_path, manifest_path, preprocess_path, checkpoint_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required artifact: {p}")

class_map = json.loads(class_map_path.read_text(encoding="utf-8"))
labels = [c["label"] for c in sorted(class_map, key=lambda x: x["id"])]
num_classes = len(labels)

preprocess = json.loads(preprocess_path.read_text(encoding="utf-8"))
input_size = int(preprocess["input_size"])
mean = preprocess["mean"]
std = preprocess["std"]

manifest = pd.read_csv(manifest_path)
manifest["class_id"] = manifest["class_id"].astype(int)
manifest["split"] = manifest["split"].astype(str).str.lower()

val_df = manifest[manifest.split == "val"].copy().reset_index(drop=True)
test_df = manifest[manifest.split == "test"].copy().reset_index(drop=True)
print("val rows:", len(val_df), "test rows:", len(test_df))


val rows: 1716 test rows: 1714


In [5]:
eval_tfms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

class EvalDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        img = Image.open(row.path).convert("RGB")
        x = eval_tfms(img)
        return x, int(row.class_id), str(row.path)

INFERENCE_BATCH_SIZE = 8  # Reduce to 1, or set inference_device to CPU, if needed.

def make_loader(frame, bs=INFERENCE_BATCH_SIZE):
    return DataLoader(EvalDataset(frame), batch_size=bs, shuffle=False, num_workers=0)

# Release a previous model when this cell is rerun after an allocation failure.
if "model" in globals():
    del model
    gc.collect()

# This trusted local checkpoint was produced by 04_fine_tuning.ipynb.
# DirectML tensors use NumPy reconstruction objects unsupported by weights_only
# in PyTorch 2.4.1. Only use this mode with checkpoints you trust.
state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
state_dict = state["state_dict"] if isinstance(state, dict) and "state_dict" in state else state

def load_eval_model(device):
    # The checkpoint supplies all weights; no ImageNet download is needed.
    net = mobilenet_v3_small(weights=None)
    net.classifier[3] = nn.Linear(net.classifier[3].in_features, num_classes)
    net.load_state_dict(state_dict)
    return net.to(device).eval()

def is_gpu_out_of_memory(exc):
    message = str(exc).lower()
    return any(text in message for text in (
        "out of memory", "not enough gpu video memory", "could not allocate tensor",
    ))

inference_device = dml  # Set torch.device("cpu") to skip DirectML entirely.
try:
    model = load_eval_model(inference_device)
except RuntimeError as exc:
    if inference_device.type == "cpu" or not is_gpu_out_of_memory(exc):
        raise
    inference_device = torch.device("cpu")
    print("GPU memory exhausted while loading the model; using CPU.")
    # Construct after the exception scope so failed GPU allocations can be released.
if inference_device.type == "cpu":
    gc.collect()
    model = load_eval_model(inference_device)

@torch.no_grad()
def forward_on_cpu(xb):
    # Only the current batch and model live on the GPU. All saved outputs are CPU tensors.
    xb = xb.to(inference_device)
    feats = model.features(xb)
    emb = torch.flatten(model.avgpool(feats), 1)
    logits = model.classifier(emb)
    return emb.cpu(), logits.cpu()

def infer_batch(xb):
    global model, inference_device
    try:
        return forward_on_cpu(xb)
    except RuntimeError as exc:
        if inference_device.type == "cpu" or not is_gpu_out_of_memory(exc):
            raise
    # Retry this same batch on CPU; earlier outputs and sample ordering are preserved.
    print("GPU memory exhausted during inference; continuing on CPU.")
    del model
    gc.collect()
    inference_device = torch.device("cpu")
    model = load_eval_model(inference_device)
    return forward_on_cpu(xb)

def collect_logits(loader):
    logits_all = []
    targets_all = []
    paths_all = []
    for xb, yb, paths in loader:
        _, logits = infer_batch(xb)
        logits_all.append(logits)
        targets_all.append(yb.cpu().numpy())
        paths_all.extend(paths)
    return torch.cat(logits_all), np.concatenate(targets_all), paths_all

val_logits, val_y, val_paths = collect_logits(make_loader(val_df))
test_logits, test_y, test_paths = collect_logits(make_loader(test_df))
print("val logits shape:", val_logits.shape)
print("test logits shape:", test_logits.shape)


val logits shape: torch.Size([1716, 9])
test logits shape: torch.Size([1714, 9])


### Step 2 - temperature scaling on validation logits

We fit a scalar temperature `T` to minimize NLL on the validation set.

In [6]:
def nll_from_logits(logits, labels, T):
    logits_t = logits / T
    log_probs = torch.log_softmax(logits_t, dim=1)
    return -(log_probs[torch.arange(labels.shape[0]), labels].mean()).item()

def fit_temperature(logits, labels, initial_T=1.0, lr=0.1, steps=200):
    T = torch.nn.Parameter(torch.tensor(float(initial_T), dtype=torch.float32))
    opt = torch.optim.LBFGS([T], lr=lr, max_iter=steps, line_search_fn="strong_wolfe")
    y = torch.tensor(labels, dtype=torch.long)

    def closure():
        opt.zero_grad()
        loss = nn.functional.cross_entropy(logits / T.clamp_min(1e-3), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(T.clamp_min(1e-3).detach().item())

val_T = fit_temperature(val_logits, val_y, initial_T=1.0, lr=0.1, steps=200)
print("fitted temperature T:", val_T)
print("val NLL before T:", nll_from_logits(val_logits, val_y, 1.0))
print("val NLL after T :", nll_from_logits(val_logits, val_y, val_T))


fitted temperature T: 1.0282130241394043
val NLL before T: 0.025662677362561226
val NLL after T : 0.025647243484854698


### Step 3 - class prototypes from the validation set

We obtain an embedding before the classifier head, then compute a prototype per class.

In [7]:
def embedding_and_logits(frame, batch_size=INFERENCE_BATCH_SIZE):
    loader = make_loader(frame, bs=batch_size)
    emb_list = []
    logits_list = []
    y_list = []
    for xb, yb, _ in loader:
        emb, logits = infer_batch(xb)
        emb_list.append(emb)
        logits_list.append(logits)
        y_list.append(yb.cpu().numpy())
    return torch.cat(emb_list, dim=0), torch.cat(logits_list, dim=0), np.concatenate(y_list)

val_emb, _, val_emb_y = embedding_and_logits(val_df)
test_emb, _, test_emb_y = embedding_and_logits(test_df)
assert np.array_equal(val_emb_y, val_y)
assert np.array_equal(test_emb_y, test_y)

missing_classes = sorted(set(range(num_classes)) - set(val_emb_y.tolist()))
if missing_classes:
    raise ValueError(f"Cannot build validation prototypes; missing class IDs: {missing_classes}")
emb_norm = nn.functional.normalize(val_emb, dim=1)
prototypes = torch.stack([
    emb_norm[val_emb_y == class_id].mean(dim=0)
    for class_id in range(num_classes)
])
prototypes = nn.functional.normalize(prototypes, dim=1)
print("prototype shape:", prototypes.shape)

# Summaries run only after embeddings and class prototypes have been created.
def multi_metric_summary(logits, y_true, emb, T, thr_margin=0.2, thr_distance=0.85):
    logits_t = logits / T
    probs = torch.softmax(logits_t, dim=1)
    conf, pred = probs.max(dim=1)
    margin = conf - probs.scatter(1, pred.unsqueeze(1), 0).max(dim=1).values
    emb_norm = torch.nn.functional.normalize(emb, dim=1)
    sims = emb_norm @ prototypes.to(emb.device).t()
    best_sim = sims.max(dim=1).values
    top_label = pred
    labels = torch.tensor(y_true, dtype=torch.long)
    ok = (pred == labels) & (conf >= 0.5) & (margin >= thr_margin) & (best_sim >= thr_distance)
    uncertain = ~ok & (conf < 0.7)
    reject = (~ok) & (~uncertain)
    return {
        "acc": (pred == labels).float().mean().item(),
        "conf_mean": conf.mean().item(),
        "margin_mean": margin.mean().item(),
        "sim_mean": best_sim.mean().item(),
        "possible_match_rate": ok.float().mean().item(),
        "uncertain_rate": uncertain.float().mean().item(),
        "reject_rate": reject.float().mean().item(),
    }

val_summary = multi_metric_summary(val_logits, val_y, val_emb, val_T)
test_summary = multi_metric_summary(test_logits, test_y, test_emb, val_T)
print("val summary:", json.dumps(val_summary, indent=2))
print("test summary:", json.dumps(test_summary, indent=2))


prototype shape: torch.Size([9, 576])
val summary: {
  "acc": 0.9906759858131409,
  "conf_mean": 0.9902803897857666,
  "margin_mean": 0.9808940291404724,
  "sim_mean": 0.8091171979904175,
  "possible_match_rate": 0.22960372269153595,
  "uncertain_rate": 0.01107226125895977,
  "reject_rate": 0.7593240141868591
}
test summary: {
  "acc": 0.9912485480308533,
  "conf_mean": 0.989421546459198,
  "margin_mean": 0.9794982671737671,
  "sim_mean": 0.8066917657852173,
  "possible_match_rate": 0.20361727476119995,
  "uncertain_rate": 0.012835472822189331,
  "reject_rate": 0.7835472822189331
}


### Step 4 - tune app thresholds on validation set

We sweep confidence and embedding-distance thresholds, then choose a conservative policy.

In [10]:
conf_grid = np.linspace(0.50, 0.95, 10)
margin_grid = np.linspace(0.05, 0.40, 8)
sim_grid = np.linspace(0.55, 0.90, 8)

results = []
for conf_thr in conf_grid:
    for margin_thr in margin_grid:
        for sim_thr in sim_grid:
            logits_t = val_logits / val_T
            probs = torch.softmax(logits_t, dim=1)
            conf, pred = probs.max(dim=1)
            second = probs.scatter(1, pred.unsqueeze(1), 0).max(dim=1).values
            margin = conf - second
            emb_norm = torch.nn.functional.normalize(val_emb, dim=1)
            sims = emb_norm @ prototypes.to(emb_norm.device).t()
            best_sim = sims.max(dim=1).values
            labels = torch.tensor(val_y, dtype=torch.long)
            keep = (pred == labels) & (conf >= conf_thr) & (margin >= margin_thr) & (best_sim >= sim_thr)
            results.append({
                "conf_thresh": float(conf_thr),
                "margin_thresh": float(margin_thr),
                "sim_thresh": float(sim_thr),
                "keep_rate": float(keep.float().mean().item()),
                "acc_on_keep": float((pred[keep] == labels[keep]).float().mean().item()) if keep.any() else 0.0,
            })

res_df = pd.DataFrame(results)
res_df = res_df.sort_values(["acc_on_keep", "keep_rate"], ascending=False).reset_index(drop=True)
res_df.head(10)


,conf_thresh,margin_thresh,sim_thresh,keep_rate,acc_on_keep
0,0.50,0.05,0.55,0.988928,1.0
1,0.50,0.10,0.55,0.988345,1.0
2,0.50,0.15,0.55,0.988345,1.0
3,0.55,0.05,0.55,0.988345,1.0
4,0.55,0.10,0.55,0.988345,1.0
5,0.55,0.15,0.55,0.988345,1.0
6,0.50,0.20,0.55,0.987762,1.0
7,0.55,0.20,0.55,0.987762,1.0
8,0.60,0.05,0.55,0.987179,1.0
9,0.60,0.10,0.55,0.987179,1.0


In [11]:
best_policy = {
    "temperature": float(val_T),
    "confidence_threshold": 0.80,
    "margin_threshold": 0.15,
    "prototype_similarity_threshold": 0.72,
    "decision_policy": {
        "possible_match": "top_logit_after_temperature_scaling and margin/confidence/prototype checks pass",
        "uncertain": "low confidence or low margin or low prototype similarity",
        "unsupported_or_unclear": "image quality issues, broad mismatch to class prototypes, or low confidence after calibration"
    }
}

(ARTIFACTS / "rejection_policy.json").write_text(json.dumps(best_policy, indent=2), encoding="utf-8")
print(json.dumps(best_policy, indent=2))


{
  "temperature": 1.0282130241394043,
  "confidence_threshold": 0.8,
  "margin_threshold": 0.15,
  "prototype_similarity_threshold": 0.72,
  "decision_policy": {
    "possible_match": "top_logit_after_temperature_scaling and margin/confidence/prototype checks pass",
    "uncertain": "low confidence or low margin or low prototype similarity",
    "unsupported_or_unclear": "image quality issues, broad mismatch to class prototypes, or low confidence after calibration"
  }
}


### Step 5 - evaluate calibration and rejection on val/test sets

In [12]:
def evaluate_policy(logits, y_true, emb, T, conf_thr=0.80, margin_thr=0.15, sim_thr=0.72):
    logits_t = logits / T
    probs = torch.softmax(logits_t, dim=1)
    conf, pred = probs.max(dim=1)
    second = probs.scatter(1, pred.unsqueeze(1), 0).max(dim=1).values
    margin = conf - second
    emb_norm = torch.nn.functional.normalize(emb, dim=1)
    sims = emb_norm @ prototypes.to(emb_norm.device).t()
    best_sim = sims.max(dim=1).values
    labels = torch.tensor(y_true, dtype=torch.long)

    keep = (pred == labels) & (conf >= conf_thr) & (margin >= margin_thr) & (best_sim >= sim_thr)
    uncertain = (conf < conf_thr) | (margin < margin_thr) | (best_sim < sim_thr)
    reject = ~(keep | uncertain)
    return {
        "accuracy_on_kept": float((pred[keep] == labels[keep]).float().mean().item()) if keep.any() else 0.0,
        "keep_rate": float(keep.float().mean().item()),
        "uncertain_rate": float(uncertain.float().mean().item()),
        "reject_rate": float(reject.float().mean().item()),
        "mean_confidence": float(conf.mean().item()),
        "mean_margin": float(margin.mean().item()),
        "mean_proto_similarity": float(best_sim.mean().item()),
    }

val_policy = evaluate_policy(val_logits, val_y, val_emb, val_T)
test_policy = evaluate_policy(test_logits, test_y, test_emb, val_T)
print("val policy:", json.dumps(val_policy, indent=2))
print("test policy:", json.dumps(test_policy, indent=2))

report = {
    "temperature": float(val_T),
    "val": val_policy,
    "test": test_policy,
    "policy": best_policy,
}
(ARTIFACTS / "rejection_calibration_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("saved", ARTIFACTS / "rejection_calibration_report.json")


val policy: {
  "accuracy_on_kept": 1.0,
  "keep_rate": 0.9108391404151917,
  "uncertain_rate": 0.08566433936357498,
  "reject_rate": 0.003496503457427025,
  "mean_confidence": 0.9902803897857666,
  "mean_margin": 0.9808940291404724,
  "mean_proto_similarity": 0.8091171979904175
}
test policy: {
  "accuracy_on_kept": 1.0,
  "keep_rate": 0.9084014296531677,
  "uncertain_rate": 0.08868144452571869,
  "reject_rate": 0.0029171528294682503,
  "mean_confidence": 0.989421546459198,
  "mean_margin": 0.9794982671737671,
  "mean_proto_similarity": 0.8066917657852173
}
saved C:\Users\baabyface\Documents\my work\agrilens\agrilens-training\artifacts\rejection_calibration_report.json
